# Silver Layer

### Import Libraries 

In [5]:
##

from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col,
    to_date,
    regexp_replace
)

from pyspark.sql.types import (
    IntegerType,
    DecimalType,
    DateType
)



StatementMeta(, c768189e-4767-4b1b-bfcf-169a98091135, 6, Finished, Available, Finished, False)

### Bronze Product 

In [10]:
# 1.1 Bronze_Product lesen
df_product = spark.read.table("Bronze_Lakehouse.dbo.Bronze_Product")


StatementMeta(, c768189e-4767-4b1b-bfcf-169a98091135, 11, Finished, Available, Finished, False)

In [55]:

# 1.2 Trasformacodes

df_product_silver = (
    df_product
    .withColumnRenamed("Standard Cost", "Standard_Cost")
    .withColumnRenamed("Background Color Format", "Background_Color_Format")
    .withColumnRenamed("Font Color Format", "Font_Color_Format")
    .fillna({
        "Product": "Unknown",
        "Color": "Unknown",
        "Subcategory": "Unknown",
        "Category": "Unknown",
        "Background_Color_Format": "Unknown",
        "Font_Color_Format": "Unknown"
    })
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_system", lit("Bronze_Layer"))
    .dropDuplicates()
)
display(df_product_silver.limit(10))


StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b440d07b-c6e6-4c69-a4ea-d3b4127753a2)

In [22]:
# 1.3 In Silver Layer speichern
df_product_silver.write.mode("overwrite").saveAsTable("Silver_Product")


StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 24, Finished, Available, Finished, False)

### Bronze Region


In [ ]:
# 2.1 Bronze_Region lesen
df_region = spark.read.table("Bronze_Lakehouse.dbo.Bronze_Region")


In [27]:
# 2.2 Transformcodes                
df_region_silver = (
    df_region
    .fillna({
        "Region": "Unknown",
        "Country": "Unknown",
        "Group": "Unknown"
    })
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("source_system", lit("Bronez_Layer"))
    .dropDuplicates()
)


display(df_region_silver.limit(10))


StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a81627ff-e88d-46e1-ac1d-b8cea055c4e6)

In [29]:
# 2.3 In Siler Layer speichern
df_region_silver.write.mode("overwrite").saveAsTable("Silver_Region")


StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 31, Finished, Available, Finished, False)

### Bronze Reseller lesen

In [ ]:
# 3.1 Bronze Reseller Lesen     
df_reseller = spark.read.table("Bronze_Lakehouse.dbo.Bronze_Reseller")


In [30]:
# 3.2 Transoformcodes
df_reseller_silver = (
    df_reseller
    .withColumnRenamed("Business Type", "Business_Type")
    .withColumnRenamed("State-Province", "State_Province")
    .withColumnRenamed("Country-Region", "Country_Region")
    .fillna({
        "Reseller": "Unknown",
        "Business_Type": "Unknown",
        "City": "Unknown",
        "State_Province": "Unknown",
        "Country_Region": "Unknown"
    })
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("Bronze_Layer"))
    .dropDuplicates()
)

display(df_reseller_silver.limit(10))



StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b07b313e-9bbd-484f-8bfd-913c05447871)

In [31]:
# 3.3 I Silver Layer speichern
df_reseller_silver.write.mode("overwrite").saveAsTable("Silver_Reseller")

StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 33, Finished, Available, Finished, False)

### Bronze Sales 

In [7]:
# 4.1 Bronze Sales Lesen
df_sales = spark.read.table("Bronze_Lakehouse.dbo.Bronze_Sales")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

StatementMeta(, c768189e-4767-4b1b-bfcf-169a98091135, 8, Finished, Available, Finished, False)

In [8]:
# 4.2 Transformcodes
df_sales_silver = (
    df_sales
    .withColumnRenamed("Unit Price", "Unit_Price")

    # Datentypen
    .withColumn("OrderDate", to_date(col("OrderDate"), "EEEE, MMMM d, yyyy"))
    .withColumn("ProductKey", col("ProductKey").cast(IntegerType()))
    .withColumn("ResellerKey", col("ResellerKey").cast(IntegerType()))
    .withColumn("EmployeeKey", col("EmployeeKey").cast(IntegerType()))
    .withColumn("SalesTerritoryKey", col("SalesTerritoryKey").cast(IntegerType()))
    .withColumn("Quantity", col("Quantity").cast(IntegerType()))

    # Währungsfelder bereinigen und casten
    .withColumn("Unit_Price",regexp_replace(regexp_replace(col("Unit_Price"), "\\$", ""),",","").cast(DecimalType(18, 2)))
    .withColumn("Sales",regexp_replace(regexp_replace(col("Sales"), "\\$", ""),",","").cast(DecimalType(18, 2)))
    .withColumn("Cost",regexp_replace(regexp_replace(col("Cost"), "\\$", ""),",","").cast(DecimalType(18, 2))
    )

    # Null Handling
    .fillna({"Quantity": 0,"Unit_Price": 0,"Sales": 0,"Cost": 0})

    # Audit Spalten
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("Bronze_Layer"))

    # Deduplikation
    .dropDuplicates()
)

display(df_sales_silver.limit(10))


StatementMeta(, c768189e-4767-4b1b-bfcf-169a98091135, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4fafb0ab-3112-43fc-a470-0788b41f3aa2)

In [43]:
# 4.3 In Silver Layer speichern
df_salesperson = spark.read.table("Bronze_Lakehouse.dbo.Bronze_Salesperson")


StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 45, Finished, Available, Finished, False)

In [44]:
df_salesperson_silver = (
    df_salesperson
    .withColumn("EmployeeKey", col("EmployeeKey").cast(IntegerType()))
    .fillna({
        "EmployeeID": "Unknown",
        "Salesperson": "Unknown",
        "Title": "Unknown",
        "UPN": "Unknown"
    })
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("Bronze_Layer"))
    .dropDuplicates()
)

display(df_salesperson_silver.limit(10))

StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 46, Finished, Available, Finished, False)

In [46]:
df_salesperson_silver.write.mode("overwrite").saveAsTable("Silver_Salesperson")

StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 48, Finished, Available, Finished, False)

  ### SalesPersonRegion

In [12]:
# 5.1 Bronze SalesPersonRegion Lesen
df_salesperson_region = spark.read.table("Bronze_Lakehouse.dbo.Bronze_SalespersonRegion")


StatementMeta(, c768189e-4767-4b1b-bfcf-169a98091135, 13, Finished, Available, Finished, False)

In [13]:
# 5.2 Transformcodes
df_salesperson_region_silver = (
    df_salesperson_region
    .withColumn("EmployeeKey", col("EmployeeKey").cast(IntegerType()))
    .withColumn("SalesTerritoryKey", col("SalesTerritoryKey").cast(IntegerType()))
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("Bronze_Layer"))
    .dropDuplicates()
)

display(df_salesperson_region_silver.limit(10))

StatementMeta(, c768189e-4767-4b1b-bfcf-169a98091135, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0d9b4c3-824d-4ff5-abe6-a641c22cfd5b)

In [50]:
# 5.3 In Silver Layer speichern
df_salesperson_region_silver.write.mode("overwrite").saveAsTable("Silver_SalespersonRegion")                

StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 52, Finished, Available, Finished, False)

### Broneze_Target 

In [51]:
# 6.1 Broneze Target Lesen
df_targets = spark.read.table("Bronze_Lakehouse.dbo.Bronze_Targets")


StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 53, Finished, Available, Finished, False)

In [56]:
# 6.2 Transformacodes
df_targets_silver = (
    df_targets
    .withColumn("EmployeeID", col("EmployeeID").cast(IntegerType()))
    .withColumn("Target", col("Target").cast(DecimalType(18,2)))
    .fillna({
        "Target": 0
    })
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("Bronze_Layer"))
    .dropDuplicates()
)

display(df_targets_silver.limit(10))

StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5a987247-4a8d-4e3a-980a-0d879accb7f6)

In [54]:
# 6.3 In Silver Layer speichern
df_targets_silver.write.mode("overwrite").saveAsTable("Silver_Targets")

StatementMeta(, dbcb5a96-1c2b-487d-b416-5257e5ba4f0b, 56, Finished, Available, Finished, False)